# Neural Networks and Deep Learning — Seminar 6: NLP foundations: text representation and word embeddings
Prof. dr. Daniel Traian Pele — Bucharest University of Economic Studies (ASE); Romanian Academy, Institute for Economic Forecasting (IPE)

[Open in Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L6_seminar.ipynb)

Seminar tasks on real data from `../data/clean` (see `../data/SOURCES.md`). Write your code in the empty cells under each task; keep the seed fixed so that your numbers can be compared.

In [ ]:
# Google Colab: copy the course repository, so that the data and the saved results are found
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Setup

In [ ]:
import re
from collections import Counter
import torch, torch.nn as nn, torch.nn.functional as F
torch.set_num_threads(4)
TOK = re.compile(r"[a-z]+(?:'[a-z]+)?|\d+(?:[.,]\d+)*")
def tok(s):
    s = re.sub(r'https?://\S+|@\w+', ' ', str(s).lower())
    return ['<num>' if t[0].isdigit() else t for t in TOK.findall(s)]
pb = pd.read_csv(DATA + 'financial_phrasebank.csv')
tw = pd.read_csv(DATA + 'twitter_fin_sentiment.csv')
sp = pd.read_csv(DATA + 'ecb_speeches_2022plus.csv', parse_dates=['date'])
LAB = ['negative', 'neutral', 'positive']
pb_y = pb.label.map({k: i for i, k in enumerate(LAB)}).values
pb_tok = [tok(s) for s in pb.sentence]
print(len(pb), len(tw), len(sp))

In [ ]:
rs = np.random.RandomState(SEED); tr, va, te = [], [], []
for c in range(3):
    ids = rs.permutation(np.where(pb_y == c)[0]); n = len(ids)
    tr += list(ids[:int(.7 * n)]); va += list(ids[int(.7 * n):int(.85 * n)]); te += list(ids[int(.85 * n):])
tr, va, te = map(np.array, (tr, va, te))
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
J = lambda ids: [' '.join(pb_tok[i]) for i in ids]
def lr_tuned(Xtr, ytr, Xva, yva, Xte):
    best = max(((f1_score(yva, LogisticRegression(C=C, max_iter=3000, class_weight='balanced').fit(Xtr, ytr).predict(Xva), average='macro'), C)
                for C in [0.1, 1.0, 10.0]))
    m = LogisticRegression(C=best[1], max_iter=3000, class_weight='balanced').fit(Xtr, ytr)
    return m.predict(Xte), m
print(len(tr), len(va), len(te))

**Task 1.** Compute by hand the TF-IDF weight (sublinear tf, smoothed idf) of a word that occurs twice in a sentence and in 30 of the 3,000 training sentences; check with `TfidfVectorizer` on a toy corpus.

In [ ]:
# your code here

**Task 2.** Remove English stop words (`stop_words='english'`) from the TF-IDF model of the lecture. What happens to the macro-F1, and which removed words carried sentiment?

In [ ]:
# your code here

**Task 3.** Train the TF-IDF + logistic model only on sentences with 100% annotator agreement and test on the same test set. Is cleaner training data worth fewer examples?

In [ ]:
# your code here

**Task 4.** Apply the tweet-sentiment model (TF-IDF + logistic trained on the tweets' train split) to the PhraseBank test sentences, mapping bearish/neutral/bullish to negative/neutral/positive. How much does the domain change cost?

In [ ]:
# your code here

**Task 5.** Build a sentiment index of ECB speeches: apply the PhraseBank TF-IDF model to every sentence of each speech, average (positive − negative) by quarter and plot it.

In [ ]:
# your code here